# SFCW Radar — Debug Data Analysis

Offline analysis of the `.npy` arrays saved by the live detection loop.

> **Note:** Run `scripts/run_sweep.py --debug` first to generate `data/` files.


## 1. Setup


In [ ]:
%matplotlib inline
import glob
import re
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import yaml

# Resolve repo paths relative to this notebook (notebooks/ -> repo root).
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CONFIG_PATH = REPO_ROOT / "config" / "radar_params.yaml"
DATA_DIR = REPO_ROOT / "data"

with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

radar = config["radar"]
F_START = float(radar["f_start"])   # 1 GHz
F_STOP = float(radar["f_stop"])     # 4 GHz
N_STEPS = int(radar["n_steps"])     # 201

# LO frequency grid (Hz) and a GHz view for plotting.
frequencies = np.linspace(F_START, F_STOP, N_STEPS)
freq_ghz = frequencies / 1e9

# Range axis (meters): 201 bins spanning ~0-10 m.
range_res_m = float(config["performance"]["range_resolution_m"])
range_axis = np.arange(N_STEPS) * range_res_m

print(f"Config loaded: {N_STEPS} steps, {F_START/1e9:.1f}-{F_STOP/1e9:.1f} GHz")
print(f"Range axis: 0 to {range_axis[-1]:.2f} m ({range_res_m:.3f} m/bin)")


## 2. Load Debug Data


In [ ]:
# Each --debug frame writes three files sharing a common base name:
#   <base>_range_profile.npy, <base>_cfar_threshold.npy, <base>_freq_spectrum.npy
# where <base> = frame_<index>_<YYYYmmdd_HHMMSS_ffffff>.

def load_frames(data_dir: Path):
    """Group debug .npy files by frame and load the three arrays for each."""
    frames = []
    for rp_path in sorted(data_dir.glob("*_range_profile.npy")):
        base = rp_path.name[: -len("_range_profile.npy")]
        thr_path = data_dir / f"{base}_cfar_threshold.npy"
        spec_path = data_dir / f"{base}_freq_spectrum.npy"
        if not (thr_path.exists() and spec_path.exists()):
            continue  # skip incomplete frame triplets

        m = re.search(r"frame_(\d+)_(\d{8}_\d{6}_\d+)", base)
        index = int(m.group(1)) if m else -1
        stamp = m.group(2) if m else ""

        frames.append({
            "index": index,
            "stamp": stamp,
            "range_profile": np.load(rp_path),
            "cfar_threshold": np.load(thr_path),
            "freq_spectrum": np.load(spec_path),
        })

    frames.sort(key=lambda fr: fr["index"])
    return frames


frames = load_frames(DATA_DIR)

if not frames:
    print("No debug frames found in", DATA_DIR)
    print("Run: python scripts/run_sweep.py --debug")
else:
    print(f"Found {len(frames)} frames")
    stamps = [fr["stamp"] for fr in frames if fr["stamp"]]
    if len(stamps) >= 2:
        t0 = datetime.strptime(stamps[0], "%Y%m%d_%H%M%S_%f")
        t1 = datetime.strptime(stamps[-1], "%Y%m%d_%H%M%S_%f")
        print(f"Time span: {stamps[0]} -> {stamps[-1]} "
              f"({(t1 - t0).total_seconds():.1f} s)")


## 3. Range Profile Viewer


In [ ]:
# Latest frame: range profile with the CFAR threshold overlaid.
if frames:
    fr = frames[-1]
    profile = fr["range_profile"]
    threshold = np.array(fr["cfar_threshold"], dtype=float)

    # Edge cells carry +inf thresholds; mask them so autoscaling behaves.
    threshold_plot = threshold.copy()
    threshold_plot[~np.isfinite(threshold_plot)] = np.nan

    detections = profile > threshold  # inf edges never exceed

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(range_axis, profile, label="range profile")
    ax.plot(range_axis, threshold_plot, "r--", label="CFAR threshold")

    if np.any(detections):
        det_idx = np.flatnonzero(detections)
        peak = det_idx[np.argmax(profile[det_idx])]
        ax.plot(range_axis[peak], profile[peak], "rv", markersize=12,
                label=f"peak {range_axis[peak]:.2f} m")

    ax.set_xlabel("Range (m)")
    ax.set_ylabel("Magnitude")
    ax.set_title(f"Range Profile \u2014 frame {fr['index']}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()
else:
    print("No frames to plot.")


## 4. B-scan (Range \u00d7 Time Waterfall)


In [ ]:
# Stack every frame's range profile into a (n_frames, n_steps) image.
if frames:
    bscan = np.vstack([fr["range_profile"] for fr in frames])

    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(
        bscan, cmap="hot", aspect="auto", origin="lower",
        extent=[range_axis[0], range_axis[-1], 0, len(frames)],
    )
    ax.set_xlabel("Range (m)")
    ax.set_ylabel("Frame index")
    ax.set_title("B-scan Waterfall")
    fig.colorbar(im, ax=ax, label="Magnitude")
    plt.show()
else:
    print("No frames to plot.")


## 5. Frequency Spectrum


In [ ]:
# Latest frame's TX spectrum (stored in dB by the pipeline).
if frames:
    spectrum_db = frames[-1]["freq_spectrum"]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(freq_ghz, spectrum_db)
    ax.axvline(1.0, color="r", linestyle="--", alpha=0.7)
    ax.axvline(4.0, color="r", linestyle="--", alpha=0.7)
    ax.set_xlabel("Frequency (GHz)")
    ax.set_ylabel("Magnitude (dB)")
    ax.set_title("TX Frequency Spectrum")
    ax.grid(True, alpha=0.3)
    plt.show()
else:
    print("No frames to plot.")


## 6. CFAR Detection Summary


In [ ]:
# Per-frame detection flag: any range bin above its CFAR threshold.
if frames:
    flags = np.array([
        bool(np.any(fr["range_profile"] > fr["cfar_threshold"]))
        for fr in frames
    ], dtype=int)
    indices = [fr["index"] for fr in frames]

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.bar(indices, flags, width=0.9)
    ax.set_xlabel("Frame index")
    ax.set_ylabel("Detected (0/1)")
    ax.set_ylim(0, 1.2)
    ax.set_yticks([0, 1])
    ax.set_title("CFAR Detection Over Time")
    plt.show()

    total = len(flags)
    detected = int(flags.sum())
    rate = 100.0 * detected / total if total else 0.0
    print(f"Total frames:    {total}")
    print(f"Detected frames: {detected}")
    print(f"Detection rate:  {rate:.1f}%")
else:
    print("No frames to analyze.")
